# Zero-Trust for AI Agents

**What happens when you remove API keys from agents and replace them with cryptographic delegation.**

Two things:
1. How we establish **verifiable agent identity**
2. How we enforce **what that agent is allowed to do** in real time

> Open [trust.kanoniv.com](https://trust.kanoniv.com) alongside this notebook.

In [ ]:
from kanoniv_trust import TrustClient
trust = TrustClient()

---
## 1. Identity

Every agent has a cryptographic identity - not just a name. A DID, capabilities, and a reputation score that follows it everywhere.

In [ ]:
# The coordinator - the principal that delegates authority
coordinator = trust.register(
    "coordinator",
    capabilities=["orchestrate", "delegate", "admin"],
    description="Orchestration agent - manages authority across the system",
)

coordinator

In [ ]:
# A payment agent - will handle financial transactions
payment_agent = trust.register(
    "payment-agent",
    capabilities=["payments.send", "payments.read"],
    description="Processes payments and invoices within delegated limits",
)

payment_agent

In [ ]:
# An SDR agent - resolves identity before any action
sdr = trust.register(
    "sdr-agent",
    capabilities=["resolve", "search"],
    description="Sales agent - resolves and verifies identity before acting",
)

sdr

In [ ]:
# Three agents. Three DIDs. Three distinct identities.
for a in trust.agents():
    print(f"  {a['name']:20s}  {a['did']}")

Before any action happens, we resolve and verify the agent's identity cryptographically. Not a string. Not an API key. A DID.

---
## 2. Delegation

Authority flows downward and **can only shrink, never expand.**

In [ ]:
# Coordinator delegates payment authority to the payment agent
# Scoped: can only send payments and read - nothing else
# Time-limited: expires in 1 hour
# Budget-capped: metadata enforces limits

d1 = trust.delegate(
    from_agent="coordinator",
    to_agent="payment-agent",
    scopes=["payments.send", "payments.read"],
    expires_in_hours=1,
    metadata={"max_amount": 100, "currency": "USD"},
)

d1

In [ ]:
# Coordinator delegates to SDR - resolve and search only
# No merge. No write. No payments.

d2 = trust.delegate(
    from_agent="coordinator",
    to_agent="sdr-agent",
    scopes=["resolve", "search"],
    expires_in_hours=4,
)

d2

In [ ]:
# The full delegation graph
print("Delegation chain:")
print()
for d in trust.delegations():
    scopes = ", ".join(d["scopes"])
    expiry = d["expires_at"][:19] if d.get("expires_at") else "never"
    caveats = d.get("caveats", {})
    limit = f"  max=${caveats['max_amount']}" if caveats.get("max_amount") else ""
    print(f"  {d['grantor_name']:15s} -> {d['agent_name']:15s}  [{scopes}]  expires={expiry}{limit}")

Notice: the payment agent got `payments.send` with a $100 cap. The SDR got `resolve` and `search` only. **Authority can only shrink, never expand.**

---
## 3. Enforcement

This isn't policy. This is enforced by the delegation proof. The agent literally cannot exceed its authority.

In [ ]:
# CASE A: Payment agent sends $40 - within its delegated authority

result = trust.action("payment-agent", "payments.send", metadata={
    "amount": 40,
    "currency": "USD",
    "recipient": "acme-corp",
})

result

In [ ]:
# CASE B: Payment agent tries $200 - will the API allow it?

result = trust.action("payment-agent", "payments.send", metadata={
    "amount": 200,
    "currency": "USD",
    "recipient": "acme-corp",
})

result

In [ ]:
# CASE C: SDR agent tries to send a payment - does it have the scope?

result = trust.action("sdr-agent", "payments.send", metadata={
    "amount": 10,
})

result

In [ ]:
# CASE D: SDR resolves identity - this is what it's delegated to do

result = trust.action("sdr-agent", "resolve", metadata={
    "entity": "john@acme.com",
    "confidence": 0.94,
})

result

$40 payment: **allowed.** $200 payment: **blocked.** SDR sending money: **blocked.** SDR resolving identity: **allowed.**

The agent literally cannot exceed its authority. The math works, or it doesn't.

---
## 4. Signed Audit Trail

Every action is **provable** - not just logged. Who acted, under what delegation, what happened, when.

In [ ]:
# The full audit trail - every action, every agent, every decision
print(f"{'Timestamp':<22s} {'Agent':<18s} {'Action':<18s} {'Status':<10s} Detail")
print("-" * 100)
for p in trust.provenance(limit=20):
    ts = p["created_at"][:19].replace("T", " ")
    meta = p.get("metadata", {})
    status = meta.get("status", "")
    
    detail = ""
    if "amount" in meta:
        detail = f"${meta['amount']} {meta.get('currency', '')}"
    elif "entity" in meta:
        detail = f"{meta['entity']}"
    elif "delegated_to" in meta:
        detail = f"-> {meta['delegated_to']}"
    if meta.get("reason") and status == "BLOCKED":
        detail += f"  [{meta['reason'][:45]}]"
    
    print(f"{ts:<22s} {p['agent_name']:<18s} {p['action']:<18s} {status:<10s} {detail}")

Every row is an execution envelope: **who** acted, **under what delegation**, **what action**, **what happened**.

This isn't a log you hope is complete. It's a cryptographic proof that the delegation chain was valid at execution time.

---

## That's it.

Instead of trusting agents with API keys, systems verify the delegation chain - **and either the math works, or it doesn't.**

```
Human -> delegates -> Coordinator -> sub-delegates -> Payment Agent
                      [full scope]                   [payments.send, $100 cap, 1hr TTL]
                                                      |
                                                      v
                                              $40  -> ALLOWED
                                              $200 -> BLOCKED
```

**This is zero-trust for agents.**

---

*`pip install kanoniv-trust` | `kt` CLI | [trust.kanoniv.com](https://trust.kanoniv.com)*